In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt


# ============================================================
# 1. System parameters
# ============================================================

# Number of gamma/beta configurations
N = 3

# Hamiltonian in the ordered basis:
#     0 -> LL
#     1 -> LR
#     2 -> RL
#     3 -> RR
#
# Replace this example with your actual Hamiltonian.
H = np.array(
    [
        [1.0, 0.1, 0.2, 0.0],
        [0.1, 2.0, 0.3, 0.2],
        [0.2, 0.3, 2.0, 0.1],
        [0.0, 0.2, 0.1, 3.0],
    ],
    dtype=complex,
)

# Optional check: a physical Hamiltonian is normally Hermitian.
if not np.allclose(H, H.conj().T):
    print("Warning: H is not Hermitian.")


# ============================================================
# 2. Basis indexing
# ============================================================

# Use:
#     L -> 0
#     R -> 1
#
# pair_index generates:
#     LL -> 0
#     LR -> 1
#     RL -> 2
#     RR -> 3

def pair_index(a, b):
    return 2 * a + b


# ============================================================
# 3. Psi equations
# ============================================================

def psi_rhs(t, f, psi, H, singular_tolerance=1e-12):
    """
    Calculate dpsi/dt for all gamma, m and v.

    Array conventions
    -----------------
    f.shape = (N,)

        f[gamma] = f_gamma(t)

    psi.shape = (N, 2, 2)

        psi[gamma, 0, 0] = psi_(gamma,1,L)
        psi[gamma, 0, 1] = psi_(gamma,1,R)
        psi[gamma, 1, 0] = psi_(gamma,2,L)
        psi[gamma, 1, 1] = psi_(gamma,2,R)

    H.shape = (4, 4)

        H is expressed in the basis LL, LR, RL, RR.

    Returns
    -------
    dpsi : complex ndarray with shape (N, 2, 2)
    """

    number_of_configurations = len(f)

    # The equation contains 1/f_gamma.
    if np.any(np.abs(f) < singular_tolerance):
        bad_indices = np.where(np.abs(f) < singular_tolerance)[0]

        raise FloatingPointError(
            "The psi equation is singular because the following "
            f"f_gamma values are close to zero: gamma={bad_indices.tolist()}. "
            "The equation explicitly contains 1/f_gamma."
        )

    # --------------------------------------------------------
    # Construct q_beta:
    #
    # q_beta = psi_(beta,1) tensor psi_(beta,2)
    #
    # q[beta] has components ordered as:
    #
    # q[beta, 0] = psi_(beta,1,L) psi_(beta,2,L)   -> LL
    # q[beta, 1] = psi_(beta,1,L) psi_(beta,2,R)   -> LR
    # q[beta, 2] = psi_(beta,1,R) psi_(beta,2,L)   -> RL
    # q[beta, 3] = psi_(beta,1,R) psi_(beta,2,R)   -> RR
    #
    # Shape: (N, 4)
    # --------------------------------------------------------

    q = np.einsum(
        "bk,bl->bkl",
        psi[:, 0, :],
        psi[:, 1, :],
    ).reshape(number_of_configurations, 4)

    # --------------------------------------------------------
    # Sum over beta:
    #
    # weighted_q = sum_beta f_beta q_beta
    #
    # Shape: (4,)
    # --------------------------------------------------------

    weighted_q = np.sum(
        f[:, np.newaxis] * q,
        axis=0,
    )

    # --------------------------------------------------------
    # Apply the Hamiltonian once:
    #
    # H_weighted_q = H @ sum_beta(f_beta q_beta)
    #
    # Shape: (4,)
    #
    # Its components are:
    #
    # H_weighted_q[0] = <LL|H|weighted_q>
    # H_weighted_q[1] = <LR|H|weighted_q>
    # H_weighted_q[2] = <RL|H|weighted_q>
    # H_weighted_q[3] = <RR|H|weighted_q>
    # --------------------------------------------------------

    H_weighted_q = H @ weighted_q

    dpsi = np.zeros_like(psi, dtype=complex)

    # --------------------------------------------------------
    # Calculate all psi derivatives
    # --------------------------------------------------------

    for gamma in range(number_of_configurations):
        for v in range(2):

            # =================================================
            # Equation for mathematical m = 1
            #
            # Stored using Python index m = 0.
            #
            # i d/dt psi_(gamma,1,v)
            #
            #   = (1/f_gamma) sum_j
            #       conjugate(psi_(gamma,2,j))
            #       <v,j|H|weighted_q>
            # =================================================

            total_m1 = 0.0j

            for j in range(2):
                vj_index = pair_index(v, j)

                total_m1 += (
                    np.conj(psi[gamma, 1, j])
                    * H_weighted_q[vj_index]
                )

            # Divide by i: 1/i = -i
            dpsi[gamma, 0, v] = (
                -1j * total_m1 / f[gamma]
            )

            # =================================================
            # Equation for mathematical m = 2
            #
            # Stored using Python index m = 1.
            #
            # i d/dt psi_(gamma,2,v)
            #
            #   = (1/f_gamma) sum_j
            #       conjugate(psi_(gamma,1,j))
            #       <j,v|H|weighted_q>
            # =================================================

            total_m2 = 0.0j

            for j in range(2):
                jv_index = pair_index(j, v)

                total_m2 += (
                    np.conj(psi[gamma, 0, j])
                    * H_weighted_q[jv_index]
                )

            dpsi[gamma, 1, v] = (
                -1j * total_m2 / f[gamma]
            )

    return dpsi


# ============================================================
# 4. Full coupled equations for f and psi
# ============================================================

def coupled_rhs(t, y, H, N):
    """
    Right-hand side of the full coupled system.

    y contains:
        [all f values, all flattened psi values]
    """

    # --------------------------------------------------------
    # Unpack the state
    # --------------------------------------------------------

    f = y[:N]

    psi = y[N:].reshape(N, 2, 2)

    # --------------------------------------------------------
    # First calculate dpsi/dt
    # --------------------------------------------------------

    dpsi = psi_rhs(t, f, psi, H)

    # --------------------------------------------------------
    # Construct q_gamma for the f equations
    #
    # q[gamma] =
    #     psi[gamma,0,:] tensor psi[gamma,1,:]
    #
    # Shape: (N, 4)
    # --------------------------------------------------------

    q = np.einsum(
        "gi,gj->gij",
        psi[:, 0, :],
        psi[:, 1, :],
    ).reshape(N, 4)

    # --------------------------------------------------------
    # V[gamma,beta] =
    #
    #     conjugate(q_gamma) @ H @ q_beta
    #
    # Shape: (N, N)
    # --------------------------------------------------------

    V = q.conj() @ H @ q.T

    # --------------------------------------------------------
    # Sum over beta:
    #
    # interaction[gamma] =
    #     sum_beta V[gamma,beta] f[beta]
    #
    # Shape: (N,)
    # --------------------------------------------------------

    interaction = V @ f

    # Equivalent explicit notation:
    #
    # interaction = np.einsum("gb,b->g", V, f)

    # --------------------------------------------------------
    # A[gamma] =
    #
    #     sum_(p,u)
    #       conjugate(psi[gamma,p,u])
    #       dpsi[gamma,p,u]/dt
    #
    # Shape: (N,)
    # --------------------------------------------------------

    A = np.sum(
        psi.conj() * dpsi,
        axis=(1, 2),
    )

    # --------------------------------------------------------
    # f equation:
    #
    # df_gamma/dt =
    #     -i sum_beta V_(gamma,beta) f_beta
    #     -A_gamma f_gamma
    #
    # V @ f sums over beta.
    # A * f is elementwise: A_gamma f_gamma.
    # --------------------------------------------------------

    df = -1j * interaction - A * f

    # Combine all derivatives into a 1D vector
    return np.concatenate(
        [
            df,
            dpsi.ravel(),
        ]
    )


# ============================================================
# 5. Initial conditions
# ============================================================

# IMPORTANT:
# The psi equation divides by every f_gamma.
# Therefore, do not initialize any f_gamma exactly equal to zero
# unless the equations have first been mathematically reformulated.

f0 = np.array(
    [1.0, 0.1, 0.1],
    dtype=complex,
)

# Normalize f0 if required by the physical model.
f0 /= np.linalg.norm(f0)

# psi0 has shape:
#     (gamma, particle, L/R)
psi0 = np.zeros((N, 2, 2), dtype=complex)

# Example: initialize every single-particle state as
#
#     (|L> + |R>)/sqrt(2)
for gamma in range(N):
    for particle in range(2):
        psi0[gamma, particle, :] = (
            np.array([1.0, 1.0], dtype=complex)
            / np.sqrt(2.0)
        )

# Combine f0 and psi0 into one state vector.
y0 = np.concatenate(
    [
        f0,
        psi0.ravel(),
    ]
)


# ============================================================
# 6. Consistency checks before integration
# ============================================================

assert H.shape == (4, 4)
assert f0.shape == (N,)
assert psi0.shape == (N, 2, 2)
assert y0.shape == (N + 4 * N,)

# Evaluate the RHS once to catch indexing errors.
test_derivative = coupled_rhs(
    t=0.0,
    y=y0,
    H=H,
    N=N,
)

assert test_derivative.shape == y0.shape
assert np.all(np.isfinite(test_derivative))

print("Initial state shape:", y0.shape)
print("Initial derivative shape:", test_derivative.shape)


# ============================================================
# 7. Numerical integration
# ============================================================

t_start = 0.0
t_end = 10.0

evaluation_times = np.linspace(
    t_start,
    t_end,
    1001,
)

solution = solve_ivp(
    fun=lambda t, y: coupled_rhs(t, y, H, N),
    t_span=(t_start, t_end),
    y0=y0,
    t_eval=evaluation_times,
    method="DOP853",
    rtol=1e-9,
    atol=1e-11,
)

if not solution.success:
    raise RuntimeError(
        f"Integration failed: {solution.message}"
    )


# ============================================================
# 8. Extract f and psi solutions
# ============================================================

# Shape:
#     (N, number_of_times)
f_solution = solution.y[:N, :]

# Shape:
#     (N, 2, 2, number_of_times)
psi_solution = solution.y[N:, :].reshape(
    N,
    2,
    2,
    len(solution.t),
)

print("f_solution shape:", f_solution.shape)
print("psi_solution shape:", psi_solution.shape)


# Individual examples:
#
# f_solution[gamma, time_index]
#
# psi_solution[gamma, particle, state, time_index]
#
# State index:
#     0 -> L
#     1 -> R
#
# Particle index:
#     0 -> particle 1
#     1 -> particle 2

f_gamma_0 = f_solution[0, :]

psi_gamma0_particle1_L = psi_solution[0, 0, 0, :]
psi_gamma0_particle1_R = psi_solution[0, 0, 1, :]
psi_gamma0_particle2_L = psi_solution[0, 1, 0, :]
psi_gamma0_particle2_R = psi_solution[0, 1, 1, :]


# ============================================================
# 9. Plot the f populations
# ============================================================

for gamma in range(N):
    plt.plot(
        solution.t,
        np.abs(f_solution[gamma, :])**2,
        label=fr"$|f_{{{gamma}}}(t)|^2$",
    )

plt.xlabel("Time")
plt.ylabel(r"$|f_\gamma(t)|^2$")
plt.legend()
plt.tight_layout()
plt.show()


# ============================================================
# 10. Plot an example psi population
# ============================================================

plt.plot(
    solution.t,
    np.abs(psi_solution[0, 0, 0, :])**2,
    label=r"$|\psi_{0,1,L}(t)|^2$",
)

plt.plot(
    solution.t,
    np.abs(psi_solution[0, 0, 1, :])**2,
    label=r"$|\psi_{0,1,R}(t)|^2$",
)

plt.xlabel("Time")
plt.ylabel("Population")
plt.legend()
plt.tight_layout()
plt.show()